# 00 – Combine AI and real CSVs

Unify the TikTok metadata exports and add helper columns (`source`, `has_frames`, `has_video`) before running the embedding notebook.


In [1]:
from pathlib import Path

import cv2
import pandas as pd


In [2]:
DATA_DIR =  Path('../../data')

AI_CSV = DATA_DIR / '01_raw/videos_metadata/01_AI_TIKTOK_VIDEOS_DATASET_2025.csv'
REAL_CSV = DATA_DIR / '01_raw/videos_metadata/01_REAL_TIKTOK_VIDEOS_DATASET_2025.csv'
OUTPUT_CSV = DATA_DIR / '01_raw/videos_metadata/01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv'

FRAME_ROOTS = {
    'ai': DATA_DIR / '02_media/ai_videos_2025/frames',
    'real': DATA_DIR / '02_media/real_videos_2025/frames',
}

VIDEO_ROOTS = {
    'ai': DATA_DIR / '02_media/ai_videos_2025/videos',
    'real': DATA_DIR / '02_media/real_videos_2025/videos',
}

# Alle Videos als MP4 in einem gemeinsamen Ordner
VIDEO_LENGTH_ROOT = DATA_DIR / '02_media/stratified_sample/videos'

print(f"AI CSV: {AI_CSV}")
print(f"Real CSV: {REAL_CSV}")
print(f"Output: {OUTPUT_CSV}")
print(f"Video-Length-Ordner: {VIDEO_LENGTH_ROOT}")


AI CSV: ..\..\data\01_raw\videos_metadata\01_AI_TIKTOK_VIDEOS_DATASET_2025.csv
Real CSV: ..\..\data\01_raw\videos_metadata\01_REAL_TIKTOK_VIDEOS_DATASET_2025.csv
Output: ..\..\data\01_raw\videos_metadata\01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv
Video-Length-Ordner: ..\..\data\02_media/stratified_sample/videos


In [3]:
def load_with_source(path: Path, source_label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['source'] = source_label
    return df

df_ai = load_with_source(AI_CSV, 'ai')
df_real = load_with_source(REAL_CSV, 'real')
combined = pd.concat([df_ai, df_real], ignore_index=True, sort=False)
print(f"Loaded {len(df_ai)} AI rows and {len(df_real)} real rows -> {len(combined)} combined")


Loaded 538 AI rows and 922 real rows -> 1460 combined


In [5]:
def has_frames(video_id: str, source: str) -> bool:
    folder = FRAME_ROOTS.get(source)
    print(f"Checking frames for video ID {video_id} in folder {folder}")
    if folder is None:
        return False
    return (folder / str(video_id)).is_dir()

def resolve_video_path(video_id: str, source: str) -> Path | None:
    source_folder = VIDEO_ROOTS.get(source)
    if source_folder is not None:
        source_path = source_folder / f"{video_id}.mp4"
        if source_path.is_file():
            return source_path

    merged_path = VIDEO_LENGTH_ROOT / f"{video_id}.mp4"
    if merged_path.is_file():
        return merged_path

    return None

def has_video(video_id: str, source: str) -> bool:
    video_path = resolve_video_path(video_id, source)
    print(f"Checking video for video ID {video_id}: {video_path}")
    return video_path is not None

def get_video_duration_seconds(video_id: str, source: str):
    video_path = resolve_video_path(video_id, source)
    if video_path is None:
        return pd.NA

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        cap.release()
        return pd.NA

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps is None or fps <= 0 or frame_count is None or frame_count < 0:
        return pd.NA

    return round(frame_count / fps, 3)

combined['has_frames'] = [
    has_frames(str(row['id']), row['source']) if 'id' in combined.columns else False
    for _, row in combined.iterrows()
]

combined['has_video'] = [
    has_video(str(row['id']), row['source']) if 'id' in combined.columns else False
    for _, row in combined.iterrows()
]

combined['video_duration_seconds'] = [
    get_video_duration_seconds(str(row['id']), row['source']) if 'id' in combined.columns else pd.NA
    for _, row in combined.iterrows()
]
combined.head()


Checking frames for video ID 7516032103390203178 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7517830042206899469 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7516243181650988334 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7516395955562859819 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7518222079955602743 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7515925552549678378 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7516376664570416426 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7521490969468865847 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7521488514597276983 in folder ..\..\data\02_media/ai_videos_2025/frames
Checking frames for video ID 7517774065457728782 in folder ..\..\data\02_media/ai_videos_20

,id,thread_id,author,author_full,author_followers,author_likes,author_videos,author_avatar,body,stickers,...,challenges,diversification_labels,location_created,effects,warning,missing_fields,source,has_frames,has_video,video_duration_seconds
0,7516032103390203178,7516032103390203178,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,let’s GO!! #ai #fyp #viral #Vlog #googleveo3 #...,NaN,...,"ai,fyp,viral,vlog,googleveo3,trending",NaN,NaN,Brownie Grain,NaN,NaN,ai,True,True,8.0
1,7517830042206899469,7517830042206899469,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,i'm Kalaï and i was literally prompted to serv...,your new favorite ai influencer 💞,...,"fyp,googleveo3,ai,aiinfluencer,viral,trending,...",NaN,NaN,NaN,NaN,NaN,ai,True,True,8.0
2,7516243181650988334,7516243181650988334,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,oop 🫢🫢 #fyp #trending #googleveo3 #ai #viral ...,🫢,...,"fyp,trending,googleveo3,ai,viral,vlog,podcast,...",NaN,NaN,Brownie Grain,NaN,NaN,ai,True,True,6.958
3,7516395955562859819,7516395955562859819,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,xoxo #loveisland #viral #vlog #fyp #trending #...,NaN,...,"loveisland,viral,vlog,fyp,trending,googleveo3,ai",NaN,NaN,Brownie Grain,NaN,NaN,ai,True,True,6.708
4,7518222079955602743,7518222079955602743,ai.kalai,kalai.ai,9773,20,55,https://p16-sign-va.tiktokcdn.com/tos-maliva-a...,PSA 🩷 #advice #bigsisteradvice #therapy #thera...,NaN,...,"advice,bigsisteradvice,therapy,therapytiktok,s...",NaN,NaN,Brownie Grain,NaN,NaN,ai,True,True,8.0


In [6]:
video_rename_map = {
    # IDs und Zeitstempel
    "id": "video_id",
    "thread_id": "video_thread_id",
    "timestamp": "video_timestamp",
    "unix_timestamp": "video_unix_timestamp",

    # Autorin und Influencer-Typ
    "author": "author_username",
    "author_full": "author_displayname",
    "author_followers": "author_follower_count",
    "author_likes": "author_like_count_total",
    "author_videos": "author_video_count",
    "author_avatar": "author_avatar_url",

    # Inhalte
    "body": "video_caption",
    "stickers": "video_stickers",
    "hashtags": "video_hashtags",
    "challenges": "video_challenges",
    "diversification_labels": "video_diversification_labels",
    "effects": "video_effects",
    "location_created": "video_location_created",

    # URLs
    "video_url": "video_file_url",
    "tiktok_url": "video_platform_url",
    "thumbnail_url": "video_thumbnail_url",

    # Musik
    "music_thumbnail": "music_thumbnail_url",

    # Flags
    "is_duet": "video_is_duet",
    "is_ad": "video_is_ad",
    "is_paid_partnership": "video_is_paid_partnership",
    "is_sensitive": "video_is_sensitive",
    "is_photosensitive": "video_is_photosensitive",
    "warning": "video_warning",
    "has_frames": "video_has_frames",
    "has_video": "video_has_video",

    # Engagement
    "likes": "video_like_count",
    "comments": "video_comment_count",
    "shares": "video_share_count",
    "plays": "video_view_count",

    # Sonstiges
    "missing_fields": "video_missing_fields",
    "source": "influencer_type",
}

videos_renamed = combined.rename(columns=video_rename_map)

videos_renamed.columns


Index(['video_id', 'video_thread_id', 'author_username', 'author_displayname',
       'author_follower_count', 'author_like_count_total',
       'author_video_count', 'author_avatar_url', 'video_caption',
       'video_stickers', 'video_timestamp', 'video_unix_timestamp',
       'video_is_duet', 'video_is_ad', 'video_is_paid_partnership',
       'video_is_sensitive', 'video_is_photosensitive', 'music_name',
       'music_id', 'music_url', 'music_thumbnail_url', 'music_author',
       'video_file_url', 'video_platform_url', 'video_thumbnail_url',
       'video_like_count', 'video_comment_count', 'video_share_count',
       'video_view_count', 'video_hashtags', 'video_challenges',
       'video_diversification_labels', 'video_location_created',
       'video_effects', 'video_warning', 'video_missing_fields',
       'influencer_type', 'video_has_frames', 'video_has_video',
       'video_duration_seconds'],
      dtype='object')

In [ ]:
videos_renamed["influencer"] = videos_renamed["author_username"].fillna(videos_renamed["author_username"])

# Whitelist der erlaubten Influencer 
allowed_influencers = [
    "ai.kalai",
    "brexleyai",
    "millas_sofia",
    "lilmiquela",
    "imma.tokyo",
    "jessicawangofficial",
    "ericanic0le",
    "hollybrandmusic",
    "nobodywhoareu",
    "misshannahashton",
]

# Namen f?r den Vergleich normalisieren
videos_renamed["influencer_clean"] = videos_renamed["influencer"].str.strip().str.lower()

allowed_clean = [name for name in allowed_influencers]

# F?lle mit Video, aber ohne Frames pr?fen
missing_frames = videos_renamed[(videos_renamed["video_has_video"] == False) & (videos_renamed["video_has_frames"] == False)]
print("Videos mit Video aber ohne Frames:", missing_frames["video_id"].nunique())
print("Video-IDs ohne Frames:", missing_frames["video_id"].unique())

# Nur Zeilen mit Frames und Videodatei behalten
videos_renamed = videos_renamed[(videos_renamed["video_has_frames"] == True) & (videos_renamed["video_has_video"] == True)].copy()

# Auf erlaubte Influencerinnen begrenzen
df_videos = videos_renamed[videos_renamed["influencer_clean"].isin(allowed_clean)].copy()

print("Anzahl Videos nach Filter:", df_videos["video_id"].nunique())
print("Anzahl Influencer nach Filter:", df_videos["influencer"].nunique())
df_videos["influencer"].value_counts()

Videos mit Video aber ohne Frames: 158
Video-IDs ohne Frames: [7554620301015797010 7558266664366787853 7494965700780461317
 7565241332218727735 7558006625169116423 7498731879420513558
 7517679263798103309 7514484998938856746 7513741471997283630
 7468875175799524654 7468434752580865326 7462197167428603179
 7473504692010634518 7532074246315494678 7554390439604833558
 7489936003340242198 7500625705613430038 7545906462413901078
 7512476261764189462 7559172730050940182 7497572936769981718
 7551930939279019286 7551387599361805590 7545462497394953474
 7544422711519382806 7544023587599158530 7538812930855308566
 7532463493098900758 7531434937229053206 7531032885868989718
 7530708686763838742 7529240562998856982 7528727634914643222
 7527754093792660758 7527264250625150230 7570769962348875030
 7475341305980128534 7511026822692408599 7526599024640609558
 7525869816633396502 7522214841767972118 7521691630332661014
 7520302837994130710 7519202496837913878 7517624246596570390
 7516510393963973910 75

influencer
ericanic0le            205
nobodywhoareu          197
misshannahashton       145
lilmiquela              95
jessicawangofficial     93
hollybrandmusic         93
imma.tokyo              56
ai.kalai                55
brexleyai               53
millas_sofia            53
Name: count, dtype: int64

In [8]:
df_videos["video_timestamp"] = pd.to_datetime(df_videos["video_timestamp"], errors="coerce")


influencers_with_2024 = ["jessicawangofficial", "imma.tokyo", "lilmiquela"]

df_filtered = df_videos[
    (
        df_videos["influencer"].isin(influencers_with_2024) &
        (df_videos["video_timestamp"].dt.year.isin([2021, 2022, 2023, 2024, 2025]))
    ) |
    (
        ~df_videos["influencer"].isin(influencers_with_2024) &
        (df_videos["video_timestamp"].dt.year == 2025)
    )
].copy()

df_filtered["video_timestamp"].dt.year.value_counts()

video_timestamp
2025    844
2023     83
2021     47
2022     41
2024     20
Name: count, dtype: int64

# Hinzufügen der Engagement Rate

In [9]:
df_filtered["video_engagement_rate"] = (
    df_filtered["video_like_count"] +
    df_filtered["video_comment_count"] +
    df_filtered["video_share_count"] 
) / df_filtered["video_view_count"].replace(0, pd.NA)

df_filtered["video_engagement_rate"].describe()   

df_filtered["influencer"].value_counts()

influencer
ericanic0le            205
nobodywhoareu          197
misshannahashton       145
jessicawangofficial     93
hollybrandmusic         93
lilmiquela              91
ai.kalai                55
brexleyai               53
millas_sofia            53
imma.tokyo              50
Name: count, dtype: int64

In [10]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(OUTPUT_CSV, index=False)
print(f"Saved combined CSV with shape {df_filtered.shape} -> {OUTPUT_CSV}")

Saved combined CSV with shape (1035, 43) -> ..\..\data\01_raw\videos_metadata\01_AI_AND_REAL_TIKTOK_VIDEOS_DATASET_2025.csv
